# Notebook 11: Does mixup really reduce overfitting without losing accuracy? Repeated paired cross-validation

Notebook 10 (one split, 3 seeds) showed that **ResNet18S + mixup** (Zhang et al., ICLR 2018, α = 0.2) cut the overfitting gap from 0.69 to 0.41
and the test loss from 0.70 to 0.55 with no loss of macro-F1. Before replacing ResNet18S `base`, this notebook checks it with the same protocol
as Notebook 09 (Bouthillier et al., MLSys 2021; Nadeau & Bengio, 2003).

### Protocol (fixed before running)
1. Same 30 folds as Notebook 09: stratified 5-fold CV × 6 repeats (seeds 42, 1–5) on the 17,000 train + validation images; the official test
   set is not used in Sections 4–6. The folds are rebuilt with the same code and seeds, and checked against the saved Notebook 09 fold indices.
2. **ResNet18S `base`** = the Notebook 09 runs (R71, R73, …, R129), already trained on these folds with these seeds.
   **ResNet18S + mixup** = 30 new runs (R140–R169) with the same images, seed, augmentation, recipe and checkpoint rule, plus mixup (α = 0.2).
3. Metrics on each held-out fold: 5-group macro-F1 (**primary**), 12-class macro-F1, |power-loss error|, NLL, ECE, overfitting gap, val-loss rise.
4. Statistics on the 30 paired differences (base − mixup): Nadeau & Bengio corrected resampled t-test, and P(base better) with a 95 % bootstrap CI (γ = 0.75).
5. **Decision rule:**
   * **Keep `base`** if it is better on 5-group macro-F1 by both tests (p < 0.05 and P(base better) significant and meaningful).
   * Otherwise **switch to mixup** if its overfitting gap is significantly smaller (corrected t-test p < 0.05).
   * Otherwise keep `base`.
6. **Final confirmation, once:** 3-seed ensembles + TTA on the official test set from saved probabilities (R62–R64 vs R131–R133): McNemar test and a
   bootstrap CI of the macro-F1 difference.

About 50 min on the RTX 4070 Ti (30 runs). Needs `results_09.csv` and `cv_probs/` from Notebook 09, and `test_probs/` from Notebooks 08 and 10.
Finished runs in `results_11.csv` are skipped if you Run All again. **Close `experiment_log.xlsx` before running.**

## 0. Imports, device, data

In [ ]:
import time, datetime, platform, json, sys, hashlib
from pathlib import Path
from copy import copy
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support)
from openpyxl import load_workbook

NOTEBOOK = "11_mixup_cv.ipynb"
REL = Path("2020-02-14_InfraredSolarModules/InfraredSolarModules")
CANDIDATES = [Path("../archive"), Path("../data/archive"), Path("archive"), Path("../../archive")]
DATA_DIR = next((r / REL for r in CANDIDATES if (r / REL / "images").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f"Data folder not found. This notebook runs in {Path.cwd()}\nLooked in:\n  "
                            + "\n  ".join(str((r / REL).resolve()) for r in CANDIDATES))

if torch.cuda.is_available():
    device = torch.device("cuda"); HARDWARE = torch.cuda.get_device_name(0)
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device("mps"); HARDWARE = "Apple GPU (mps)"
else:
    device = torch.device("cpu"); HARDWARE = platform.processor() or "CPU"

for d in ("figures", "checkpoints", "runs", "test_probs", "cv_probs"):
    Path(d).mkdir(exist_ok=True)
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("device:", device, "|", HARDWARE, "| CUDA available:", torch.cuda.is_available(), "| CUDA", torch.version.cuda)
print("data:", DATA_DIR.resolve())

## 1. Split, groups and images (identical to Notebook 05, but **no resize**)

In [ ]:
split = pd.read_csv("splits.csv")
SPLIT_MD5 = hashlib.md5(Path("splits.csv").read_bytes()).hexdigest()
GROUP = {"Cell": "Cell-level", "Cell-Multi": "Cell-level", "Cracking": "Cell-level",
         "Hot-Spot": "Cell-level", "Hot-Spot-Multi": "Cell-level",
         "Diode": "Substring-level", "Diode-Multi": "Substring-level",
         "Offline-Module": "Module-level",
         "Shadowing": "External", "Vegetation": "External", "Soiling": "External",
         "No-Anomaly": "No-Anomaly"}
CLASSES = sorted(split.label.unique())
c2i = {c: i for i, c in enumerate(CLASSES)}
Y = split.label.map(c2i).values
PARTS = {s: np.where(split.split.values == s)[0] for s in ("train", "val", "test")}

t0 = time.time()
imgs = np.stack([np.asarray(Image.open(DATA_DIR / p), dtype=np.float32) / 255.0 for p in split.path])
med = np.median(imgs, axis=(1, 2), keepdims=True)
std = imgs.std(axis=(1, 2), keepdims=True) + 1e-6
Z = torch.from_numpy((imgs - med) / std).unsqueeze(1)          # (20000, 1, 40, 24): this is ALL the preprocessing
print("images:", tuple(Z.shape), f"loaded in {time.time() - t0:.1f}s | splits.csv MD5 {SPLIT_MD5}")

In [ ]:
class IRModules(Dataset):
    # training only: flips, shift +-2 px (replicate padding, crop back to 40x24), optional Gaussian noise
    def __init__(self, idx, augment=False, max_shift=2, noise_std=0.0):
        self.idx = torch.as_tensor(idx); self.y = torch.as_tensor(Y[idx]).long()
        self.augment, self.s, self.noise = augment, max_shift, noise_std
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = Z[self.idx[i]]
        if self.augment:
            if torch.rand(1) < 0.5: x = torch.flip(x, dims=[2])
            if torch.rand(1) < 0.5: x = torch.flip(x, dims=[1])
            s = self.s
            if s > 0:
                x = F.pad(x.unsqueeze(0), (s, s, s, s), mode="replicate").squeeze(0)
                dy, dx = torch.randint(0, 2 * s + 1, (2,)).tolist()
                x = x[:, dy:dy + 40, dx:dx + 24]
            if self.noise > 0:
                x = x + self.noise * torch.randn_like(x)
        return x, self.y[i]

def make_loaders(noise_std=0.0, batch=128, seed=42):
    tr = PARTS["train"]
    counts = np.bincount(Y[tr], minlength=len(CLASSES))
    w = counts[Y[tr]] ** -0.5                                         # sampler p ~ 1/sqrt(n), as R10
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(tr),
                                    replacement=True, generator=g)
    train_dl = DataLoader(IRModules(tr, augment=True, noise_std=noise_std), batch_size=batch, sampler=sampler)
    # fixed 3,000 training images, no augmentation, natural class mix: measures the training loss on the
    # same terms as the validation loss, so the train/validation gap is a fair overfitting measure
    rng = np.random.default_rng(0)
    tr_eval = DataLoader(IRModules(rng.choice(tr, 3000, replace=False)), batch_size=512)
    val_dl = DataLoader(IRModules(PARTS["val"]), batch_size=512)
    test_dl = DataLoader(IRModules(PARTS["test"]), batch_size=512)
    return train_dl, tr_eval, val_dl, test_dl, counts

## 2. The six architectures
All of them take the 40 × 24 × 1 image directly. The comments give the size of the feature map after each stage.

In [ ]:
class ResNet18S(nn.Module):
    # ResNet-18 adapted to small images (He et al. 2016, CIFAR-10 setting): conv1 3x3 stride 1, no max-pool
    # 40x24 -> layer1 40x24 -> layer2 20x12 -> layer3 10x6 -> layer4 5x3 -> global avg-pool -> fc
    def __init__(self, n_classes, pretrained=True, dropout=0.0):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        w7 = net.conv1.weight.data.clone()                             # (64, 3, 7, 7) pretrained
        net.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        if pretrained:                                                 # centre 3x3 of each 7x7 filter, RGB summed
            net.conv1.weight.data.copy_(w7[:, :, 2:5, 2:5].sum(dim=1, keepdim=True))
        net.maxpool = nn.Identity()
        net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, n_classes))
        self.net = net
    def forward(self, x):
        return self.net(x)

class ResNet10S(ResNet18S):
    # same as ResNet18S but only the FIRST basic block of each stage is kept (8 conv layers + fc instead of 16 + fc)
    def __init__(self, n_classes, pretrained=True, dropout=0.0):
        super().__init__(n_classes, pretrained, dropout)
        for name in ("layer1", "layer2", "layer3", "layer4"):
            setattr(self.net, name, nn.Sequential(getattr(self.net, name)[0]))

class WideBasic(nn.Module):
    # pre-activation block of Wide ResNet: BN-ReLU-conv3x3 -> (dropout) -> BN-ReLU-conv3x3, plus shortcut
    def __init__(self, c_in, c_out, stride, dropout):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(c_in), nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(c_out), nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.short = None if (stride == 1 and c_in == c_out) else nn.Conv2d(c_in, c_out, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.short is None else self.short(o)
        o = self.conv1(o)
        o = self.conv2(self.drop(F.relu(self.bn2(o))))
        return o + sc

class WRN(nn.Module):
    # Wide ResNet (Zagoruyko & Komodakis 2016). WRN-16-4: 3 groups x 2 blocks, widths 64/128/256
    # 40x24 -> group1 40x24 -> group2 20x12 -> group3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=16, k=4, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        widths = [16, 16 * k, 32 * k, 64 * k]
        self.conv = nn.Conv2d(1, widths[0], 3, 1, 1, bias=False)
        blocks, c_in = [], widths[0]
        for g, stride in enumerate([1, 2, 2]):
            for i in range(n):
                blocks.append(WideBasic(c_in, widths[g + 1], stride if i == 0 else 1, dropout))
                c_in = widths[g + 1]
        self.blocks = nn.Sequential(*blocks)
        self.bn = nn.BatchNorm2d(c_in)
        self.fc = nn.Linear(c_in, n_classes)
    def forward(self, x):
        o = F.relu(self.bn(self.blocks(self.conv(x))))
        return self.fc(F.adaptive_avg_pool2d(o, 1).flatten(1))

def conv_bn(c_in, c_out, k=3):
    return nn.Sequential(nn.Conv2d(c_in, c_out, k, padding=k // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU())

class MSCNN(nn.Module):
    # multi-scale CNN, idea of Korkmaz & Acikgoz 2022 (three branches with different kernel sizes);
    # simplified re-implementation. branches 3x3 / 5x5 / 7x7 at 40x24 -> concat 96 ch
    # -> 128 ch, pool 20x12 -> 128 ch, pool 10x6 -> 256 ch -> global avg-pool -> fc
    def __init__(self, n_classes, dropout=0.0):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(conv_bn(1, 32, k), conv_bn(32, 32, k)) for k in (3, 5, 7)])
        self.body = nn.Sequential(conv_bn(96, 128), nn.MaxPool2d(2), conv_bn(128, 128), nn.MaxPool2d(2),
                                  conv_bn(128, 256))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        o = torch.cat([b(x) for b in self.branches], dim=1)
        o = self.body(o)
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class CifarBasic(nn.Module):
    # basic residual block of the CIFAR ResNets (He et al. 2016)
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out)
        self.c2, self.b2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out) else
                      nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)))
        return F.relu(self.b2(self.c2(o)) + self.short(x))

class ResNetCifar(nn.Module):
    # ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 blocks, widths 16/32/64
    # 40x24 -> stage1 40x24 -> stage2 20x12 -> stage3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=20, dropout=0.0):
        super().__init__()
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, c_in = [], 16
        for c_out, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(CifarBasic(c_in, c_out, stride if i == 0 else 1)); c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        o = self.layers(self.stem(x))
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class DenseLayer(nn.Module):
    # DenseNet-BC bottleneck layer: BN-ReLU-conv1x1 (4k) -> BN-ReLU-conv3x3 (k), output concatenated to its input
    def __init__(self, c_in, k):
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm2d(c_in), nn.ReLU(), nn.Conv2d(c_in, 4 * k, 1, bias=False),
                               nn.BatchNorm2d(4 * k), nn.ReLU(), nn.Conv2d(4 * k, k, 3, 1, 1, bias=False))
    def forward(self, x):
        return torch.cat([x, self.f(x)], dim=1)

class DenseNetBC(nn.Module):
    # DenseNet-BC for small images (Huang et al. 2017), L = 100, k = 12, compression 0.5
    # 40x24 -> block1 40x24 -> transition 20x12 -> block2 -> transition 10x6 -> block3 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=100, k=12, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        c = 2 * k
        mods = [nn.Conv2d(1, c, 3, 1, 1, bias=False)]
        for b in range(3):
            for _ in range(n):
                mods.append(DenseLayer(c, k)); c += k
            if b < 2:
                c_out = c // 2
                mods += [nn.BatchNorm2d(c), nn.ReLU(), nn.Conv2d(c, c_out, 1, bias=False), nn.AvgPool2d(2)]
                c = c_out
        mods += [nn.BatchNorm2d(c), nn.ReLU()]
        self.features = nn.Sequential(*mods)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(c, n_classes))
    def forward(self, x):
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1))

def build_model(name, n_classes, dropout):
    return {"ResNet18S": lambda: ResNet18S(n_classes, pretrained=True, dropout=dropout),
            "ResNet10S": lambda: ResNet10S(n_classes, pretrained=True, dropout=dropout),
            "ResNet18S_scratch": lambda: ResNet18S(n_classes, pretrained=False, dropout=dropout),
            "WRN16_4": lambda: WRN(n_classes, 16, 4, dropout=dropout),
            "MSCNN": lambda: MSCNN(n_classes, dropout=dropout),
            "ResNet20": lambda: ResNetCifar(n_classes, 20, dropout=dropout),
            "DenseNetBC100": lambda: DenseNetBC(n_classes, 100, 12, dropout=dropout)}[name]()

MODEL_INFO = {
    "ResNet10S": "ResNet-10 = ResNet18S with 1 basic block per stage (first block of each ImageNet-pretrained stage kept), small-image stem "
                 "(conv1 3x3 s1, no max-pool), 1-channel 40x24 input, all layers fine-tuned",
    "ResNet18S": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool; He et al. 2016 CIFAR setting), 1-channel 40x24 input, "
                 "layer1-4 ImageNet-pretrained (IMAGENET1K_V1), conv1 from centre 3x3 of pretrained 7x7 (RGB summed), all layers fine-tuned",
    "ResNet18S_scratch": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool), 1-channel 40x24 input, random initialisation",
    "WRN16_4": "Wide ResNet-16-4 (Zagoruyko & Komodakis 2016), pre-activation blocks, widths 64/128/256, 1-channel 40x24 input, random initialisation",
    "ResNet20": "ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 basic blocks, widths 16/32/64, 1-channel 40x24 input, random initialisation",
    "DenseNetBC100": "DenseNet-BC L=100 k=12 for small images (Huang et al. 2017), compression 0.5, 1-channel 40x24 input, random initialisation",
    "MSCNN": "Multi-scale CNN (idea of Korkmaz & Acikgoz 2022, simplified): branches 3x3/5x5/7x7 (32 ch each) -> 128 -> pool -> 128 -> pool -> 256 -> GAP -> fc, random initialisation",
}
LR = {"ResNet18S": 3e-4, "ResNet10S": 3e-4, "ResNet18S_scratch": 1e-3, "WRN16_4": 1e-3, "MSCNN": 1e-3, "ResNet20": 1e-3, "DenseNetBC100": 1e-3}       # pretrained: smaller steps
EPOCHS = {"ResNet18S": 30, "ResNet10S": 30, "ResNet18S_scratch": 40, "WRN16_4": 40, "MSCNN": 40, "ResNet20": 40, "DenseNetBC100": 40}           # from scratch: more epochs
RECIPES = {
    "base": dict(label_smoothing=0.0, dropout=0.0, weight_decay=1e-4, noise_std=0.0),
    "reg":  dict(label_smoothing=0.1, dropout=0.3, weight_decay=5e-4, noise_std=0.05),
    "reg_noLS": dict(label_smoothing=0.0, dropout=0.3, weight_decay=5e-4, noise_std=0.05),
}
USED = ["ResNet18S", "ResNet10S"]

# self-test: shapes, parameters and one training step per model (nothing is logged)
for name in USED:
    m = build_model(name, 12, 0.3).to(device)
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
    x = Z[:8].to(device); y = torch.as_tensor(Y[:8]).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    loss = F.cross_entropy(m.train()(x), y); loss.backward(); opt.step()
    with torch.no_grad():
        out = m.eval()(x)
    print(f"{name:18s} parameters = {n_par:>10,}   input {tuple(x.shape)} -> output {tuple(out.shape)}   one step OK")
del m, opt

## 3. One function that trains, tests and logs a run
Same as Notebook 05, plus:
* the **training-subset loss** (3,000 fixed training images, no augmentation) is computed every epoch with **plain cross-entropy**, like the validation loss (label smoothing raises the training loss on purpose, so the smoothed loss is not used for the comparison);
* new overfitting numbers per run: lowest validation loss and its epoch, validation loss at the last epoch, **rise** = last − lowest (0 means no rise), and **gap** = validation loss − training-subset loss at the last epoch.

In [ ]:
def environment():
    return {"python": sys.version.split()[0], "pytorch": torch.__version__, "torchvision": torchvision.__version__,
            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
            "device": str(device), "hardware": HARDWARE, "os": platform.platform(),
            "data_dir": str(DATA_DIR.resolve()), "splits_md5": SPLIT_MD5}

def logged_ids():
    ids = set()
    for f in ("results_05.csv", "results_06.csv", "results_08.csv", "results_09.csv", "results_10.csv", "results_11.csv"):
        if Path(f).exists():
            ids |= set(pd.read_csv(f).run_id.astype(str))
    if Path("experiment_log.xlsx").exists():
        ws = load_workbook("experiment_log.xlsx", read_only=True)["Runs"]
        ids |= {str(r[0]) for r in ws.iter_rows(min_row=2, values_only=True) if r and r[0]}
    return ids

def next_free_id(run_id):
    ids = logged_ids()
    if run_id not in ids:
        return run_id
    k = 1
    while f"{run_id}-rerun{k}" in ids:
        k += 1
    return f"{run_id}-rerun{k}"

def evaluate(model, dl):
    model.eval(); ys, ps, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in dl:
            out = model(x.to(device))
            loss_sum += F.cross_entropy(out, y.to(device), reduction="sum").item()      # plain CE, no smoothing
            ps.append(out.argmax(1).cpu()); ys.append(y)
    yt, pt = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return yt, pt, loss_sum / len(yt)

K_OF = {"No-Anomaly": 0, "Diode": 1, "Diode-Multi": 2, "Offline-Module": 3}   # bypassed substrings per module (others 0)
K_ARR = np.array([K_OF.get(c, 0) for c in CLASSES])

def predict_probs(model, idx, tta=False):
    # softmax probabilities on images idx; tta = mean over original, H flip, V flip, H+V flip
    model.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(idx), 512):
            x = Z[idx[b:b + 512]].to(device)
            views = [x] + ([torch.flip(x, [3]), torch.flip(x, [2]), torch.flip(x, [2, 3])] if tta else [])
            out.append(torch.stack([F.softmax(model(v), 1) for v in views]).mean(0).cpu())
    return torch.cat(out).numpy()

def ece_score(probs, yt, bins=15):
    conf, pred = probs.max(1), probs.argmax(1); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == yt[m]).mean() - conf[m].mean())
    return float(e)

def metrics_from_probs(probs, yt):
    names = np.array(CLASSES); pt = probs.argmax(1)
    kt, kp = K_ARR[yt], K_ARR[pt]
    return dict(acc=accuracy_score(yt, pt), f1=f1_score(yt, pt, average="macro"),
                f1_5=f1_score([GROUP[c] for c in names[yt]], [GROUP[c] for c in names[pt]], average="macro"),
                fault_acc=float(np.mean((names[yt] == "No-Anomaly") == (names[pt] == "No-Anomaly"))),
                nll=float(-np.mean(np.log(np.clip(probs[np.arange(len(yt)), yt], 1e-12, 1)))),
                ece=ece_score(probs, yt), k_acc=float(np.mean(kt == kp)),
                ploss_err_pct=float(100 * (kp.sum() - kt.sum()) / kt.sum()))

def _replace_rows(ws, run_id, rows):
    for r in range(ws.max_row, 1, -1):
        if ws.cell(r, 1).value == run_id:
            ws.delete_rows(r)
    style_row = ws.max_row
    for values in rows:
        ws.append(values); r = ws.max_row
        for c in range(1, len(values) + 1):
            src, dst = ws.cell(style_row, c), ws.cell(r, c)
            if src.has_style:
                dst.font, dst.border, dst.alignment = copy(src.font), copy(src.border), copy(src.alignment)
                dst.number_format = src.number_format

def log_run(res, params, note=""):
    path = Path("experiment_log.xlsx")
    if not path.exists():
        print("experiment_log.xlsx not found: results not logged"); return
    wb = load_workbook(path)
    _replace_rows(wb["Runs"], res["run_id"], [[
        res["run_id"], datetime.date.today().isoformat(), NOTEBOOK, res["model"] + f" [{res['recipe']}]",
        "Raptor Maps, 12 classes", "splits.csv (70/15/15, seed 42)", "Raw pixels, per-image z-score, native 40x24 (no resize)",
        round(res["val_acc"], 3), round(res["val_f1"], 3), round(res["test_acc"], 3), round(res["test_f1"], 3),
        round(res["fault_acc"], 3), round(res["train_time"], 1), HARDWARE,
        (note + "; " if note else "") + f"best epoch {res['best_epoch']}; min val loss {res['val_loss_min']:.3f} @ ep {res['val_loss_min_epoch']}; "
        f"early-stopping ckpt: test F1 {res['es_f1']:.3f}, NLL {res['es_nll']:.3f}"]])
    _replace_rows(wb["Parameters"], res["run_id"], [[res["run_id"], "Run", k, v] for k, v in params.items()])
    wb.save(path)
    print(f"{res['run_id']} logged in experiment_log.xlsx")

def run_experiment(run_id, model_name, recipe, seed=42, batch=128, max_shift=2, note=""):
    rc = RECIPES[recipe]; lr, epochs = LR[model_name], EPOCHS[model_name]
    torch.manual_seed(seed); np.random.seed(seed)
    rerun_of = None
    free_id = next_free_id(run_id)
    if free_id != run_id:
        rerun_of, run_id = run_id, free_id
        note = f"Re-run of {rerun_of} with identical settings (original kept)" + (f"; {note}" if note else "")
    started = datetime.datetime.now()
    run_dir = Path("runs") / f"{run_id}_{started:%Y%m%d-%H%M%S}"
    run_dir.mkdir(parents=True, exist_ok=True)
    logf = open(run_dir / "train.log", "w", encoding="utf-8")
    def log(msg=""):
        line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"
        print(line); logf.write(line + "\n"); logf.flush()

    train_dl, tr_eval, val_dl, test_dl, counts = make_loaders(rc["noise_std"], batch, seed)
    train_dl.dataset.s = max_shift
    model = build_model(model_name, len(CLASSES), rc["dropout"]).to(device)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = nn.CrossEntropyLoss(label_smoothing=rc["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=rc["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt = Path("checkpoints") / f"{run_id}_{model_name}_{recipe}.pt"                 # best validation macro-F1
    ckpt_es = Path("checkpoints") / f"{run_id}_{model_name}_{recipe}_minloss.pt"     # lowest validation loss (early stopping)

    log(f"===== {run_id}: {model_name}, recipe = {recipe}, seed {seed}, input 1x40x24 (no resize) =====")
    log(f"folder {run_dir} | {n_par:,} parameters | lr {lr:g} | epochs {epochs} | batch {batch} | {rc} | {HARDWARE}")
    hist, best, best_ep, best_acc, t0 = [], -1.0, 0, 0.0, time.time()
    best_vl, es_ep = float("inf"), 0
    for ep in range(1, epochs + 1):
        model.train(); total, correct, seen = 0.0, 0, 0
        lr_now = optimizer.param_groups[0]["lr"]
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x); loss = criterion(out, y)
            loss.backward(); optimizer.step()
            total += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); seen += len(y)
        scheduler.step()
        ye, pe, tr_loss = evaluate(model, tr_eval)
        yv, pv, val_loss = evaluate(model, val_dl)
        f1v, accv = f1_score(yv, pv, average="macro"), accuracy_score(yv, pv)
        hist.append({"epoch": ep, "lr": lr_now, "train_loss_batches": total / seen, "train_acc_batches": correct / seen,
                     "train_eval_loss": tr_loss, "train_eval_acc": accuracy_score(ye, pe),
                     "val_loss": val_loss, "val_acc": accv, "val_macroF1": f1v, "seconds": round(time.time() - t0, 1)})
        if f1v > best:
            best, best_ep, best_acc = f1v, ep, accv
            torch.save(model.state_dict(), ckpt)
        if val_loss < best_vl:
            best_vl, es_ep = val_loss, ep
            torch.save(model.state_dict(), ckpt_es)
        log(f"epoch {ep:2d} | lr {lr_now:.2e} | train-subset loss {tr_loss:.3f} | val loss {val_loss:.3f} acc {accv:.3f} "
            f"macro-F1 {f1v:.3f}{' *' if ep == best_ep else ''}{' [min loss]' if ep == es_ep else ''} | {time.time() - t0:.0f}s")
    train_time = time.time() - t0
    h = pd.DataFrame(hist); h.to_csv(run_dir / "history.csv", index=False)

    model.load_state_dict(torch.load(ckpt, map_location=device))
    yt, pt, test_loss = evaluate(model, test_dl)
    names = np.array(CLASSES)
    i_min = int(h.val_loss.idxmin())
    res = dict(run_id=run_id, rerun_of=rerun_of, model=model_name, recipe=recipe, seed=seed,
               val_acc=best_acc, val_f1=best, best_epoch=best_ep,
               test_acc=accuracy_score(yt, pt), test_f1=f1_score(yt, pt, average="macro"),
               test_f1_as_5=f1_score([GROUP[c] for c in names[yt]], [GROUP[c] for c in names[pt]], average="macro"),
               fault_acc=float(np.mean((names[yt] == "No-Anomaly") == (names[pt] == "No-Anomaly"))),
               test_loss=test_loss,
               val_loss_min=float(h.val_loss.min()), val_loss_min_epoch=int(h.epoch[i_min]),
               val_loss_last=float(h.val_loss.iloc[-1]), val_loss_at_best=float(h.val_loss[h.epoch == best_ep].iloc[0]),
               val_loss_rise=float(h.val_loss.iloc[-1] - h.val_loss.min()),
               gap_last=float(h.val_loss.iloc[-1] - h.train_eval_loss.iloc[-1]),
               train_time=train_time, params=n_par)
    te = PARTS["test"]
    for sel, path in (("f1sel", ckpt), ("es", ckpt_es)):
        model.load_state_dict(torch.load(path, map_location=device))
        pr = predict_probs(model, te); pr_tta = predict_probs(model, te, tta=True)
        np.save(Path("test_probs") / f"nb08_{run_id}_{sel}.npy", pr)
        np.save(Path("test_probs") / f"nb08_{run_id}_{sel}_tta.npy", pr_tta)
        for k, v in metrics_from_probs(pr, yt).items():
            res[f"{sel}_{k}"] = v
    res["es_epoch"] = es_ep
    model.load_state_dict(torch.load(ckpt, map_location=device))
    p, r, f, s = precision_recall_fscore_support(yt, pt, labels=range(len(CLASSES)), zero_division=0)
    per_class = pd.DataFrame({"class": CLASSES, "precision": p, "recall": r, "f1": f, "test_images": s}).round(3)
    per_class.to_csv(run_dir / "test_per_class.csv", index=False)
    cm_counts = confusion_matrix(yt, pt, labels=range(len(CLASSES)))
    pd.DataFrame(cm_counts, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]).to_csv(run_dir / "confusion_matrix.csv")
    pd.DataFrame({"path": split.path.values[PARTS["test"]], "true": names[yt], "pred": names[pt]}).to_csv(run_dir / "test_predictions.csv", index=False)

    log(f"best epoch {best_ep} | val macro-F1 {best:.3f} | lowest val loss {res['val_loss_min']:.3f} @ epoch {res['val_loss_min_epoch']} "
        f"| last val loss {res['val_loss_last']:.3f} (rise {res['val_loss_rise']:.3f}, gap {res['gap_last']:.3f}) | {train_time:.0f}s")
    log(f"TEST accuracy {res['test_acc']:.3f} | macro-F1 {res['test_f1']:.3f} | as 5 groups {res['test_f1_as_5']:.3f} | fault/no-fault {res['fault_acc']:.3f}")
    log(f"EARLY STOPPING (min val loss, epoch {es_ep}): TEST accuracy {res['es_acc']:.3f} | macro-F1 {res['es_f1']:.3f} | "
        f"5 groups {res['es_f1_5']:.3f} | NLL {res['es_nll']:.3f} (best-F1 checkpoint {res['f1sel_nll']:.3f}) | "
        f"power-loss error {res['es_ploss_err_pct']:+.1f}% (best-F1 checkpoint {res['f1sel_ploss_err_pct']:+.1f}%)")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.4]})
    ax[0].plot(h.epoch, h.train_eval_loss, color="#2E5E8C", lw=2, label="train subset")
    ax[0].plot(h.epoch, h.val_loss, color="#8A1F2B", lw=2, label="validation")
    ax[0].axvline(res["val_loss_min_epoch"], color="#888", ls=":", lw=1)
    ax[0].set_title("Loss (plain cross-entropy)"); ax[0].set_xlabel("epoch"); ax[0].legend(frameon=False)
    ax[1].plot(h.epoch, h.val_macroF1, color="#8A1F2B", lw=2); ax[1].axvline(best_ep, color="#888", ls="--", lw=1)
    ax[1].set_title("Validation macro-F1"); ax[1].set_xlabel("epoch")
    cm = confusion_matrix(yt, pt, labels=range(len(CLASSES)), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax[2], cmap="Reds", values_format=".2f", colorbar=False)
    ax[2].tick_params(axis="x", rotation=60); ax[2].set_title(f"{run_id}: test confusion matrix")
    for a in ax[:2]: a.grid(alpha=0.25)
    plt.tight_layout(); plt.savefig(run_dir / "curves_and_cm.png", dpi=200)
    plt.savefig(f"figures/{run_id}_{model_name}_{recipe}.png", dpi=200); plt.close(fig)

    config = {"run_id": run_id, "rerun_of": rerun_of, "note": note, "started": started.isoformat(timespec="seconds"),
              "notebook": NOTEBOOK, "run_folder": str(run_dir), "checkpoint": str(ckpt),
              "data": {"dataset": "Raptor Maps InfraredSolarModules (2020-02-14)", "split_file": "splits.csv", "classes": CLASSES,
                       "train_images_per_class": dict(zip(CLASSES, counts.tolist()))},
              "input": "1 x 40 x 24, per-image z = (I - median) / (std + 1e-6), NO resize, NO channel repeat",
              "augmentation": {"hflip_p": 0.5, "vflip_p": 0.5, "random_shift_px": max_shift, "shift_padding": "replicate",
                               "gaussian_noise_std": rc["noise_std"], "applied_to": "training set only"},
              "balancing": "WeightedRandomSampler, p = 1/sqrt(n_class), 14,000 draws per epoch, replacement",
              "model": {"name": model_name, "description": MODEL_INFO[model_name], "trainable_parameters": n_par},
              "training": {"recipe": recipe, "loss": f"CrossEntropyLoss(label_smoothing={rc['label_smoothing']})",
                           "dropout": rc["dropout"], "optimizer": "AdamW", "lr": lr, "weight_decay": rc["weight_decay"],
                           "lr_schedule": f"CosineAnnealingLR(T_max={epochs})", "epochs": epochs, "batch_train": batch,
                           "batch_eval": 512, "seed": seed, "model_selection": "two checkpoints: best validation macro-F1 (main) and lowest validation loss (early stopping)",
                           "overfit_monitor": "plain CE on 3,000 fixed training images (no augmentation) and on validation"},
              "environment": environment(),
              "results": {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in res.items()}}
    config["results"]["finished"] = datetime.datetime.now().isoformat(timespec="seconds")
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    params = {
        "Run folder": str(run_dir), "Classes": f"12 ({', '.join(CLASSES)})",
        "Input": "1 x 40 x 24 native, per-image z-score; no resize, no channel repeat",
        "Balancing": "WeightedRandomSampler, p ∝ 1/sqrt(n_class), 14,000 draws per epoch, replacement",
        "Augmentation": f"H flip p=0.5, V flip p=0.5, shift ±{max_shift} px (replicate), Gaussian noise σ={rc['noise_std']}; training only",
        "Model": MODEL_INFO[model_name], "Trainable parameters": f"{n_par:,}",
        "Recipe": recipe, "Loss": f"Cross-entropy, label smoothing {rc['label_smoothing']}", "Dropout": str(rc["dropout"]),
        "Optimizer": f"AdamW, lr = {lr:g}, weight_decay = {rc['weight_decay']:g}",
        "LR schedule": f"CosineAnnealingLR, T_max = {epochs}", "Epochs / batch": f"{epochs} / {batch} (eval batch 512)",
        "Model selection": f"best validation macro-F1 → epoch {best_ep}; early stopping (lowest validation loss) → epoch {es_ep}",
        "Early-stopping checkpoint, test": (f"acc {res['es_acc']:.3f}, macro-F1 {res['es_f1']:.3f}, 5 groups {res['es_f1_5']:.3f}, "
                                            f"NLL {res['es_nll']:.3f}, ECE {res['es_ece']:.3f}, power-loss error {res['es_ploss_err_pct']:+.1f}%"),
        "Overfitting": (f"min val loss {res['val_loss_min']:.3f} @ epoch {res['val_loss_min_epoch']}; last {res['val_loss_last']:.3f}; "
                        f"rise {res['val_loss_rise']:.3f}; gap (val − train subset) {res['gap_last']:.3f}"),
        "Random seed": f"{seed} (torch, numpy, sampler; data split always seed 42)", "splits.csv MD5": SPLIT_MD5,
        "Environment": f"Python {sys.version.split()[0]}, PyTorch {torch.__version__}, CUDA {torch.version.cuda}, {device} ({HARDWARE}), {platform.platform()}",
        "Test macro-F1 scored as 5 groups": f"{res['test_f1_as_5']:.3f}",
        "Test precision / recall / F1 per class": "; ".join(f"{row['class']} {row.precision}/{row.recall}/{row.f1}" for _, row in per_class.iterrows()),
    }
    log_run(res, params, note)
    log(f"saved: {run_dir}"); logf.close()
    return res

RES_COLS = ["run_id", "rerun_of", "model", "recipe", "seed", "val_f1", "best_epoch", "test_acc", "test_f1", "test_f1_as_5", "fault_acc",
            "test_loss", "val_loss_min", "val_loss_min_epoch", "val_loss_at_best", "val_loss_last", "val_loss_rise", "gap_last", "train_time", "params",
            "es_epoch"] + [f"{sel}_{k}" for sel in ("f1sel", "es") for k in ("acc", "f1", "f1_5", "fault_acc", "nll", "ece", "k_acc", "ploss_err_pct")]
def save_result(r):
    new = pd.DataFrame([r]).reindex(columns=RES_COLS)
    old = pd.read_csv("results_08.csv").reindex(columns=RES_COLS) if Path("results_08.csv").exists() else pd.DataFrame(columns=RES_COLS)
    pd.concat([old[~old.run_id.isin(new.run_id)], new], ignore_index=True).to_csv("results_08.csv", index=False)

## 4. Folds (checked against Notebook 09) and the paired run function with mixup

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
POOL = np.concatenate([PARTS["train"], PARTS["val"]])          # 17,000 images; the official test set is never used here
REPEATS, K = [42, 1, 2, 3, 4, 5], 5
FOLDS = []                                                        # (repeat seed, fold, train idx, inner-val idx, held-out idx)
for rs in REPEATS:
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=rs)
    for f, (a, b) in enumerate(skf.split(POOL, Y[POOL])):
        tr_all, ho = POOL[a], POOL[b]
        tr, iv = train_test_split(tr_all, test_size=0.15, stratify=Y[tr_all], random_state=rs * 10 + f)
        FOLDS.append((rs, f, np.sort(tr), np.sort(iv), np.sort(ho)))
assert not set(POOL) & set(PARTS["test"])
for rs, f, tr, iv, ho in FOLDS:                                     # same folds as Notebook 09?
    p = Path("cv_probs") / f"rep{rs}_fold{f}_idx.npy"
    assert p.exists() and np.array_equal(np.load(p), ho), f"fold mismatch: repeat {rs}, fold {f}"
print("all 30 folds identical to Notebook 09")
print(len(FOLDS), "folds | train / inner-val / held-out sizes:", len(FOLDS[0][2]), len(FOLDS[0][3]), len(FOLDS[0][4]))

def make_cv_loaders(tr, iv, noise_std=0.0, batch=128, seed=42):
    counts = np.bincount(Y[tr], minlength=len(CLASSES))
    w = counts[Y[tr]] ** -0.5
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(tr), replacement=True, generator=g)
    train_dl = DataLoader(IRModules(tr, augment=True, noise_std=noise_std), batch_size=batch, sampler=sampler)
    rng = np.random.default_rng(0)
    tr_eval = DataLoader(IRModules(rng.choice(tr, min(3000, len(tr)), replace=False)), batch_size=512)
    val_dl = DataLoader(IRModules(iv), batch_size=512)
    return train_dl, tr_eval, val_dl, counts

CV_COLS = ["run_id", "model", "repeat_seed", "fold", "best_epoch", "val_f1", "acc", "f1", "f1_5", "fault_acc", "nll", "ece",
           "k_acc", "ploss_err_pct", "val_loss_min", "val_loss_last", "val_loss_rise", "gap_last", "train_time", "params"]

MIXUP_ALPHA = 0.2
def run_cv(run_id, model_name, rs, f, tr, iv, ho, batch=128):
    rc = RECIPES["base"]; lr, epochs = LR[model_name], EPOCHS[model_name]
    torch.manual_seed(rs); np.random.seed(rs); rng = np.random.default_rng(rs)
    started = datetime.datetime.now()
    run_dir = Path("runs") / f"{run_id}_{started:%Y%m%d-%H%M%S}"; run_dir.mkdir(parents=True, exist_ok=True)
    logf = open(run_dir / "train.log", "w", encoding="utf-8")
    def log(msg=""):
        line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"; print(line); logf.write(line + "\n"); logf.flush()
    train_dl, tr_eval, val_dl, counts = make_cv_loaders(tr, iv, rc["noise_std"], batch, rs)
    model = build_model(model_name, len(CLASSES), rc["dropout"]).to(device)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = nn.CrossEntropyLoss(label_smoothing=rc["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=rc["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt = Path("checkpoints") / f"{run_id}_{model_name}_mixup_cv.pt"
    log(f"===== {run_id}: {model_name} + mixup (alpha {MIXUP_ALPHA}), CV repeat seed {rs}, fold {f + 1}/{K} | train {len(tr)}, inner-val {len(iv)}, held-out {len(ho)} | {HARDWARE} =====")
    hist, best, best_ep, t0 = [], -1.0, 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            lam = float(rng.beta(MIXUP_ALPHA, MIXUP_ALPHA)); perm = torch.randperm(len(y), device=device)
            out = model(lam * x + (1 - lam) * x[perm])
            loss = lam * F.cross_entropy(out, y) + (1 - lam) * F.cross_entropy(out, y[perm])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        scheduler.step()
        _, _, tr_loss = evaluate(model, tr_eval)
        yv, pv, val_loss = evaluate(model, val_dl)
        f1v = f1_score(yv, pv, average="macro")
        hist.append({"epoch": ep, "train_eval_loss": tr_loss, "val_loss": val_loss, "val_macroF1": f1v, "seconds": round(time.time() - t0, 1)})
        if f1v > best:
            best, best_ep = f1v, ep; torch.save(model.state_dict(), ckpt)
        log(f"epoch {ep:2d} | train-subset loss {tr_loss:.3f} | inner-val loss {val_loss:.3f} macro-F1 {f1v:.3f}{' *' if ep == best_ep else ''} | {time.time() - t0:.0f}s")
    train_time = time.time() - t0
    h = pd.DataFrame(hist); h.to_csv(run_dir / "history.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    pr = predict_probs(model, ho)
    np.save(Path("cv_probs") / f"{run_id}.npy", pr)
    m = metrics_from_probs(pr, Y[ho])
    res = dict(run_id=run_id, model=model_name + "_mixup", repeat_seed=rs, fold=f, best_epoch=best_ep, val_f1=best, **m,
               val_loss_min=float(h.val_loss.min()), val_loss_last=float(h.val_loss.iloc[-1]),
               val_loss_rise=float(h.val_loss.iloc[-1] - h.val_loss.min()),
               gap_last=float(h.val_loss.iloc[-1] - h.train_eval_loss.iloc[-1]), train_time=train_time, params=n_par)
    log(f"HELD-OUT FOLD: acc {m['acc']:.3f} | macro-F1 {m['f1']:.3f} | 5 groups {m['f1_5']:.3f} | NLL {m['nll']:.3f} | "
        f"power-loss error {m['ploss_err_pct']:+.1f}% | gap {res['gap_last']:.3f} | {train_time:.0f}s")
    config = {"run_id": run_id, "started": started.isoformat(timespec="seconds"), "notebook": NOTEBOOK, "run_folder": str(run_dir),
              "protocol": f"mixup alpha {MIXUP_ALPHA}; repeated stratified {K}-fold CV on train+val (17,000 images), repeat seed {rs}, fold {f}; "
                          "official test set NOT used; checkpoint by inner-val macro-F1; checkpoint deleted after scoring",
              "sizes": {"train": len(tr), "inner_val": len(iv), "held_out_fold": len(ho)},
              "model": {"name": model_name, "description": MODEL_INFO[model_name], "trainable_parameters": n_par},
              "training": {"recipe": "base", "optimizer": "AdamW", "lr": lr, "weight_decay": rc["weight_decay"], "epochs": epochs,
                           "lr_schedule": f"CosineAnnealingLR(T_max={epochs})", "batch": batch, "seed": rs,
                           "balancing": "WeightedRandomSampler p ∝ 1/sqrt(n_class)", "augmentation": "H/V flip p=0.5, shift ±2 px"},
              "environment": environment(),
              "results": {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in res.items()}}
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    log_run(dict(run_id=run_id, model=model_name, recipe=f"mixup, CV rep{rs} fold{f + 1}", val_acc=float("nan"), val_f1=best, test_acc=m["acc"],
                 test_f1=m["f1"], fault_acc=m["fault_acc"], train_time=train_time, best_epoch=best_ep,
                 val_loss_min=res["val_loss_min"], val_loss_min_epoch=int(h.epoch[h.val_loss.idxmin()]), es_f1=float("nan"), es_nll=float("nan")),
            {"Run folder": str(run_dir), "Protocol": config["protocol"], "Model": MODEL_INFO[model_name], "Trainable parameters": f"{n_par:,}",
             "Optimizer": f"AdamW, lr = {lr:g}, weight_decay = {rc['weight_decay']:g}", "Epochs / batch": f"{epochs} / {batch}",
             "Random seed": str(rs), "Held-out fold result": f"acc {m['acc']:.3f}, macro-F1 {m['f1']:.3f}, 5 groups {m['f1_5']:.3f}, NLL {m['nll']:.3f}, power-loss error {m['ploss_err_pct']:+.1f}%",
             "Environment": f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, {device} ({HARDWARE})"},
            note="NB11 mixup CV (pairs with NB09 base runs); 'test' = held-out CV fold, official test set unused")
    ckpt.unlink(missing_ok=True)
    log(f"saved: {run_dir}"); logf.close()
    return res

## 5. Run the 30 mixup runs (R140–R169)

In [ ]:
done = set(pd.read_csv("results_11.csv").run_id) if Path("results_11.csv").exists() else set()
n, t_all = 140, time.time()
for rs, f, tr, iv, ho in FOLDS:
    rid = f"R{n}"; n += 1
    if rid in done:
        continue
    r = run_cv(rid, "ResNet18S", rs, f, tr, iv, ho)
    old = pd.read_csv("results_11.csv") if Path("results_11.csv").exists() else pd.DataFrame(columns=CV_COLS)
    pd.concat([old, pd.DataFrame([r]).reindex(columns=CV_COLS)], ignore_index=True).to_csv("results_11.csv", index=False)
print(f"all done in {(time.time() - t_all) / 60:.1f} min")

## 6. Paired statistics and the decision
`d` = base − mixup on the same fold. For |power-loss error|, NLL, ECE, gap and rise, lower is better.

In [ ]:
from scipy import stats
c9, c11 = pd.read_csv("results_09.csv"), pd.read_csv("results_11.csv")
A = c9[c9.model == "ResNet18S"].copy(); B = c11.copy()
for t_ in (A, B): t_["ploss_abs"] = t_.ploss_err_pct.abs()
A, B = A.set_index(["repeat_seed", "fold"]).sort_index(), B.set_index(["repeat_seed", "fold"]).sort_index()
pairs = A.index.intersection(B.index); A, B = A.loc[pairs], B.loc[pairs]
n_tr = len(FOLDS[0][2]) + len(FOLDS[0][3]); n_te = len(FOLDS[0][4]); J = len(pairs)

def corrected_t(d):
    d = np.asarray(d, float); var = d.var(ddof=1)
    t = d.mean() / np.sqrt((1 / J + n_te / n_tr) * var) if var > 0 else (0.0 if d.mean() == 0 else np.inf)
    return t, 2 * stats.t.sf(abs(t), J - 1)
def p_better(a, b, higher_is_better=True, n_boot=2000, seed=0):
    a, b = np.asarray(a, float), np.asarray(b, float)
    win = ((a > b) if higher_is_better else (a < b)).astype(float) + 0.5 * (a == b)
    rng = np.random.default_rng(seed); boots = [win[rng.integers(0, len(win), len(win))].mean() for _ in range(n_boot)]
    return win.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

rows = []
for col, hib, label in [("f1_5", True, "5-group macro-F1 (primary)"), ("f1", True, "12-class macro-F1"), ("ploss_abs", False, "|power-loss error| %"),
                        ("nll", False, "held-out NLL"), ("ece", False, "ECE"), ("gap_last", False, "overfitting gap"), ("val_loss_rise", False, "val-loss rise")]:
    d = A[col].values - B[col].values; t, p = corrected_t(d); pb, lo, hi = p_better(A[col], B[col], hib)
    rows.append(dict(metric=label, base=A[col].mean(), mixup=B[col].mean(), mean_diff=d.mean(), sd_diff=d.std(ddof=1),
                     corrected_t=t, p_value=p, P_base_better=pb, ci_low=lo, ci_high=hi))
st = pd.DataFrame(rows).round(4); st.to_csv("results_11_stats.csv", index=False)
pd.set_option("display.width", 250); print(f"{J} paired folds\n"); print(st.to_string(index=False))
f15, gap = st.iloc[0], st[st.metric == "overfitting gap"].iloc[0]
base_better = (f15.p_value < 0.05) and (f15.mean_diff > 0) and (f15.ci_low > 0.5) and (f15.ci_high > 0.75)
mix_less_overfit = (gap.p_value < 0.05) and (gap.mean_diff > 0)
if base_better:
    decision = "KEEP base: it is better than mixup on 5-group macro-F1 by both tests."
elif mix_less_overfit:
    decision = "SWITCH to ResNet18S + mixup: base is not reliably better on 5-group macro-F1, and mixup overfits significantly less."
else:
    decision = "KEEP base: no significant evidence to change."
print("\nDECISION (rule fixed before the runs):", decision)
Path("results_11_decision.txt").write_text(decision + "\n\n" + st.to_string(index=False), encoding="utf-8")

## 7. Final confirmation on the official test set (3-seed ensembles + TTA, done once)

In [ ]:
from scipy.stats import binomtest
yt = Y[PARTS["test"]]
def ens(files): return np.mean([np.load(Path("test_probs") / f) for f in files], axis=0)
Pb = ens([f"nb08_{r}_f1sel_tta.npy" for r in ("R62", "R63", "R64")]); Pm = ens([f"nb10_{r}_tta.npy" for r in ("R131", "R132", "R133")])
mb, mm = metrics_from_probs(Pb, yt), metrics_from_probs(Pm, yt)
pb_, pm_ = Pb.argmax(1), Pm.argmax(1); cb, cm = pb_ == yt, pm_ == yt
b, c = int(np.sum(cb & ~cm)), int(np.sum(~cb & cm)); p_mc = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
rng = np.random.default_rng(0); diffs = []
for _ in range(1000):
    i = rng.integers(0, len(yt), len(yt))
    diffs.append(f1_score(yt[i], pm_[i], average="macro") - f1_score(yt[i], pb_[i], average="macro"))
fin = pd.DataFrame([dict(model="ResNet18S base ensemble+TTA", **mb), dict(model="ResNet18S mixup ensemble+TTA", **mm)]).round(4)
fin.to_csv("results_11_final_test.csv", index=False); print(fin.to_string(index=False))
print(f"\nMcNemar: base only correct {b}, mixup only correct {c}, p = {p_mc:.4f}")
print(f"macro-F1 difference (mixup - base): {mm['f1'] - mb['f1']:+.3f}, 95% bootstrap CI [{np.percentile(diffs, 2.5):+.3f}, {np.percentile(diffs, 97.5):+.3f}]")
Path("results_11_final_test.txt").write_text(fin.to_string(index=False) + f"\n\nMcNemar base-only {b}, mixup-only {c}, p = {p_mc:.4f}\n"
    f"macro-F1 diff (mixup - base) {mm['f1'] - mb['f1']:+.4f}, 95% CI [{np.percentile(diffs, 2.5):+.4f}, {np.percentile(diffs, 97.5):+.4f}]\n", encoding="utf-8")

## 8. Figure for the report: `figures/fig11_mixup_cv.png`

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for a, (col, label, title) in zip(ax, [("f1_5", "5-group macro-F1 (primary)", "5-group macro-F1 (higher = better)"),
                                        ("nll", "held-out NLL", "held-out loss, NLL (lower = better)"),
                                        ("gap_last", "overfitting gap", "overfitting gap (lower = better)")]):
    for i in range(J):
        a.plot([0, 1], [A[col].values[i], B[col].values[i]], color="#BBBBBB", lw=0.8, zorder=1)
    a.scatter(np.zeros(J), A[col], color="#8A1F2B", zorder=2, label="ResNet18S base")
    a.scatter(np.ones(J), B[col], color="#3E9B6B", zorder=2, label="ResNet18S + mixup")
    row = st[st.metric == label].iloc[0]
    a.set_xticks([0, 1]); a.set_xticklabels(["base", "+ mixup"]); a.set_xlim(-0.4, 1.4); a.grid(alpha=0.25)
    a.set_title(f"{title}\ncorrected t p = {row.p_value:.3g}, P(base better) = {row.P_base_better:.2f}", fontsize=10)
ax[0].legend(frameon=False)
plt.tight_layout(); plt.savefig("figures/fig11_mixup_cv.png", dpi=200); plt.show()

## References
* H. Zhang, M. Cisse, Y. N. Dauphin, D. Lopez-Paz, "mixup: Beyond Empirical Risk Minimization," ICLR 2018.
* X. Bouthillier et al., "Accounting for Variance in Machine Learning Benchmarks," MLSys 2021.
* C. Nadeau and Y. Bengio, "Inference for the Generalization Error," Machine Learning 52, 239–281, 2003.
* T. G. Dietterich, "Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms," Neural Computation 10(7), 1998.